Import Libraries and Load Dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

FILE = "/content/FreshBasket_Loyalty_Churn_Dataset.xlsx"

assert os.path.exists(FILE), "Dataset file not found!"

profile = pd.read_excel(FILE, sheet_name="fb Customer Profile", header=1)
activity = pd.read_excel(FILE, sheet_name="fb Monthly Activity", header=1)
labels = pd.read_excel(FILE, sheet_name="fb Churn Label", header=1)

print("Profile:", profile.shape)
print("Activity:", activity.shape)
print("Labels:", labels.shape)

Data Quality Assessment

In [ ]:
for name, df in {
    "Profile": profile,
    "Activity": activity,
    "Labels": labels
}.items():
    print(f"\n{name} ")
    print("Shape:", df.shape)
    print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0])
    print("Duplicate rows:", df.duplicated().sum())

print("\nDuplicate profile IDs:",
      profile["CUSTOMER_ID"].duplicated().sum())

print("Duplicate label IDs:",
      labels["CUSTOMER_ID"].duplicated().sum())

print("Duplicate customer-month pairs:",
      activity.duplicated(["CUSTOMER_ID", "MONTH"]).sum())


print("\nChurn distribution:")
print(labels["CHURNED"].value_counts(dropna=False))

Data Validation and Consistency Checks

In [ ]:

print("Invalid ages:", ((profile["AGE"] < 18) | (profile["AGE"] > 100)).sum())
print("Negative transactions:", (activity["TRANSACTIONS"] < 0).sum())
print("Negative spend:", (activity["TOTAL_SPEND"] < 0).sum())
print("Invalid promo percentages:",
      ((activity["PROMO_TXN_PCT"] < 0) |
       (activity["PROMO_TXN_PCT"] > 1)).sum())

print("\nActivity period:",
      activity["MONTH"].min(), "to", activity["MONTH"].max())

print("Signup period:",
      profile["SIGNUP_DATE"].min(), "to", profile["SIGNUP_DATE"].max())

print("\nInsufficient history:")
print(labels["INSUFFICIENT_HISTORY_FLAG"].value_counts(dropna=False))

for col in ["GENDER", "MEMBERSHIP_TIER", "CITY", "HOME_STORE_PRICE_TIER"]:
    print(f"\n{col}:")
    print(profile[col].value_counts(dropna=False))

Investigate Data Quality Issues

In [ ]:

print("Duplicate profile IDs:",
      profile["CUSTOMER_ID"].duplicated().sum())

print("Duplicate label IDs:",
      labels["CUSTOMER_ID"].duplicated().sum())


duplicate_activity = activity[
    activity.duplicated(["CUSTOMER_ID", "MONTH"], keep=False)
].sort_values(["CUSTOMER_ID", "MONTH"])

print("\nDuplicate customer-month records:")
display(duplicate_activity)

negative_spend = activity[activity["TOTAL_SPEND"] < 0]

print("\nNegative spending records:", len(negative_spend))
display(negative_spend)

label_conflicts = labels.groupby("CUSTOMER_ID")["CHURNED"].nunique()
print("\nCustomers with conflicting churn labels:",
      (label_conflicts > 1).sum())

Investigate Duplicate Records and Negative Spending

In [ ]:

import pandas as pd
import numpy as np

profile_clean = profile.copy()
activity_clean = activity.copy()
labels_clean = labels.copy()

print("== BEFORE CLEANING ==")
print("Profile:", profile_clean.shape)
print("Activity:", activity_clean.shape)
print("Labels:", labels_clean.shape)


profile_clean = profile_clean.drop_duplicates().copy()
activity_clean = activity_clean.drop_duplicates().copy()
labels_clean = labels_clean.drop_duplicates().copy()

profile_text_cols = [
    "GENDER",
    "CITY",
    "MEMBERSHIP_TIER",
    "PREFERRED_CATEGORY",
    "HOME_STORE_PRICE_TIER"
]

for col in profile_text_cols:
    profile_clean[col] = (
        profile_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

profile_clean["GENDER"] = profile_clean["GENDER"].str.title()
profile_clean["CITY"] = profile_clean["CITY"].str.title()
profile_clean["MEMBERSHIP_TIER"] = (
    profile_clean["MEMBERSHIP_TIER"].str.title()
)
profile_clean["PREFERRED_CATEGORY"] = (
    profile_clean["PREFERRED_CATEGORY"].str.title()
)
profile_clean["HOME_STORE_PRICE_TIER"] = (
    profile_clean["HOME_STORE_PRICE_TIER"].str.title()
)

print("\n===== AFTER CLEANING =====")
print("Profile:", profile_clean.shape)
print("Activity:", activity_clean.shape)
print("Labels:", labels_clean.shape)

print("\nDuplicate profile IDs:",
      profile_clean["CUSTOMER_ID"].duplicated().sum())

print("Duplicate label IDs:",
      labels_clean["CUSTOMER_ID"].duplicated().sum())

print("Duplicate customer-month pairs:",
      activity_clean.duplicated(["CUSTOMER_ID", "MONTH"]).sum())

negative_count = (activity_clean["TOTAL_SPEND"] < 0).sum()

print("\nNegative-spend rows retained:", negative_count)
print("Total negative spend:",
      round(activity_clean.loc[
          activity_clean["TOTAL_SPEND"] < 0, "TOTAL_SPEND"
      ].sum(), 2))

print("\n===== STANDARDIZED MEMBERSHIP TIERS =====")
print(profile_clean["MEMBERSHIP_TIER"].value_counts(dropna=False))

print("\n===== STANDARDIZED CITIES =====")
print(sorted(profile_clean["CITY"].dropna().unique()))

print("\n===== MISSING VALUES AFTER CLEANING =====")
print("Profile:")
print(profile_clean.isna().sum())

print("\nActivity:")
print(activity_clean.isna().sum())

print("\nLabels:")
print(labels_clean.isna().sum())


Investigate Remaining Duplicate Monthly Records

In [ ]:

duplicate_pairs = activity_clean[
    activity_clean.duplicated(
        ["CUSTOMER_ID", "MONTH"], keep=False
    )
].sort_values(["CUSTOMER_ID", "MONTH"])

print("Duplicate customer-month rows:", len(duplicate_pairs))
display(duplicate_pairs)

Churn Distribution Analysis

In [ ]:

churn_counts = labels_clean["CHURNED"].value_counts().sort_index()

print("Churn counts:")
print(churn_counts)

print("\nChurn percentages:")
print((churn_counts / len(labels_clean) * 100).round(2))

sns.countplot(data=labels_clean, x="CHURNED")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

Resolve Duplicate Customer-Month **Records**

In [ ]:

activity_model = activity_clean.copy()

duplicate_mask = activity_model.duplicated(
    ["CUSTOMER_ID", "MONTH"], keep=False
)

activity_model["DUPLICATE_MONTH_FLAG"] = duplicate_mask.astype(int)

sum_cols = [
    "TRANSACTIONS",
    "TOTAL_SPEND",
    "APP_SESSIONS",
    "EMAILS_OPENED",
    "COUPONS_REDEEMED",
    "SUPPORT_TICKETS"
]


first_cols = [
    "DISTINCT_CATEGORIES",
    "PROMO_TXN_PCT",
    "COMPLAINT_FLAG"
]

agg = {col: "sum" for col in sum_cols}
agg.update({col: "first" for col in first_cols})
agg["DUPLICATE_MONTH_FLAG"] = "max"

activity_model = (
    activity_model
    .groupby(["CUSTOMER_ID", "MONTH"], as_index=False)
    .agg(agg)
)

activity_model["AVG_BASKET_VALUE"] = np.where(
    activity_model["TRANSACTIONS"] > 0,
    activity_model["TOTAL_SPEND"] / activity_model["TRANSACTIONS"],
    0.0
)

print("Rows before:", len(activity_clean))
print("Rows after:", len(activity_model))
print(
    "Remaining duplicate customer-month pairs:",
    activity_model.duplicated(["CUSTOMER_ID", "MONTH"]).sum()
)
print(
    "Flagged months:",
    activity_model["DUPLICATE_MONTH_FLAG"].sum()
)

display(
    activity_model[
        activity_model["DUPLICATE_MONTH_FLAG"] == 1
    ]
)

Historical Customer Behaviour by Churn Status

In [ ]:
history = activity_model[
    activity_model["MONTH"] < "2024-04-01"
].copy()


customer_behaviour = history.groupby("CUSTOMER_ID").agg(
    HIST_TRANSACTIONS=("TRANSACTIONS", "sum"),
    HIST_SPEND=("TOTAL_SPEND", "sum"),
    HIST_APP_SESSIONS=("APP_SESSIONS", "sum"),
    HIST_EMAILS_OPENED=("EMAILS_OPENED", "sum"),
    HIST_SUPPORT_TICKETS=("SUPPORT_TICKETS", "sum")
).reset_index()

eda_data = customer_behaviour.merge(
    labels_clean[["CUSTOMER_ID", "CHURNED"]],
    on="CUSTOMER_ID",
    how="inner",
    validate="one_to_one"
)

display(
    eda_data.groupby("CHURNED")[
        ["HIST_TRANSACTIONS", "HIST_SPEND",
         "HIST_APP_SESSIONS", "HIST_EMAILS_OPENED",
         "HIST_SUPPORT_TICKETS"]
    ].mean().round(2)
)

sns.boxplot(data=eda_data, x="CHURNED", y="HIST_TRANSACTIONS")
plt.title("Historical Transactions by Churn Status")
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Transactions before April 2024")
plt.tight_layout()
plt.show()

In [ ]:

metrics = [
    "HIST_TRANSACTIONS",
    "HIST_SPEND",
    "HIST_APP_SESSIONS",
    "HIST_EMAILS_OPENED",
    "HIST_SUPPORT_TICKETS"
]

display(
    eda_data.groupby("CHURNED")[metrics]
    .mean()
    .round(2)
)

plot_metrics = [
    ("HIST_APP_SESSIONS", "Historical App Sessions"),
    ("HIST_EMAILS_OPENED", "Historical Emails Opened"),
    ("HIST_SUPPORT_TICKETS", "Historical Support Tickets"),
    ("HIST_SPEND", "Historical Total Spend")
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, (column, title) in zip(axes.flat, plot_metrics):
    sns.boxplot(data=eda_data, x="CHURNED", y=column, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Churn (0 = No, 1 = Yes)")

plt.tight_layout()
plt.show()

**Create Leakage-Safe Modeling Dataset**

In [ ]:

cutoff = pd.Timestamp("2024-04-01")

hist = activity_model[activity_model["MONTH"] < cutoff].copy()

customer_features = hist.groupby("CUSTOMER_ID").agg(
    HIST_TRANSACTIONS=("TRANSACTIONS", "sum"),
    HIST_TOTAL_SPEND=("TOTAL_SPEND", "sum"),
    HIST_APP_SESSIONS=("APP_SESSIONS", "sum"),
    HIST_EMAILS_OPENED=("EMAILS_OPENED", "sum"),
    HIST_COUPONS_REDEEMED=("COUPONS_REDEEMED", "sum"),
    HIST_SUPPORT_TICKETS=("SUPPORT_TICKETS", "sum"),
    HIST_COMPLAINT_MONTHS=("COMPLAINT_FLAG", "sum"),
    HIST_DISTINCT_CATEGORIES_MEAN=("DISTINCT_CATEGORIES", "mean"),
    HIST_PROMO_TXN_PCT_MEAN=("PROMO_TXN_PCT", "mean"),
    HIST_ACTIVE_MONTHS=("TRANSACTIONS", lambda x: (x > 0).sum()),
    HIST_MONTHS_RECORDED=("MONTH", "nunique")
).reset_index()

last_purchase = (
    hist[hist["TRANSACTIONS"] > 0]
    .groupby("CUSTOMER_ID")["MONTH"]
    .max()
    .rename("LAST_ACTIVE_MONTH")
    .reset_index()
)

customer_features = customer_features.merge(
    last_purchase, on="CUSTOMER_ID", how="left",
    validate="one_to_one"
)

customer_features["RECENCY_DAYS"] = (
    cutoff - customer_features["LAST_ACTIVE_MONTH"]
).dt.days

recent = hist[hist["MONTH"] >= "2024-01-01"]

recent_features = recent.groupby("CUSTOMER_ID").agg(
    RECENT3_TRANSACTIONS=("TRANSACTIONS", "sum"),
    RECENT3_SPEND=("TOTAL_SPEND", "sum"),
    RECENT3_APP_SESSIONS=("APP_SESSIONS", "sum"),
    RECENT3_EMAILS_OPENED=("EMAILS_OPENED", "sum"),
    RECENT3_SUPPORT_TICKETS=("SUPPORT_TICKETS", "sum")
).reset_index()

customer_features = customer_features.merge(
    recent_features, on="CUSTOMER_ID", how="left",
    validate="one_to_one"
)

previous = hist[
    (hist["MONTH"] >= "2023-10-01") &
    (hist["MONTH"] < "2024-01-01")
]

previous_features = previous.groupby("CUSTOMER_ID").agg(
    PREV3_TRANSACTIONS=("TRANSACTIONS", "sum"),
    PREV3_SPEND=("TOTAL_SPEND", "sum"),
    PREV3_APP_SESSIONS=("APP_SESSIONS", "sum"),
    PREV3_EMAILS_OPENED=("EMAILS_OPENED", "sum"),
    PREV3_SUPPORT_TICKETS=("SUPPORT_TICKETS", "sum")
).reset_index()

customer_features = customer_features.merge(
    previous_features, on="CUSTOMER_ID", how="left",
    validate="one_to_one"
)

period_cols = [
    col for col in customer_features.columns
    if col.startswith(("RECENT3_", "PREV3_"))
]

customer_features[period_cols] = customer_features[period_cols].fillna(0)

profile_features = profile_clean.copy()
profile_features["TENURE_DAYS"] = (
    cutoff - profile_features["SIGNUP_DATE"]
).dt.days

profile_features = profile_features.drop(columns=["SIGNUP_DATE"])

customer_features = customer_features.merge(
    profile_features, on="CUSTOMER_ID", how="left",
    validate="one_to_one"
)

label_cols = [
    "CUSTOMER_ID", "CHURNED", "INSUFFICIENT_HISTORY_FLAG",
    "OBSERVATION_END_DATE", "LAST_PURCHASE_DATE", "MONTHS_OBSERVED"
]

modeling_df = customer_features.merge(
    labels_clean[label_cols],
    on="CUSTOMER_ID", how="inner",
    validate="one_to_one"
)

modeling_df = modeling_df[
    (modeling_df["INSUFFICIENT_HISTORY_FLAG"] == 0) &
    modeling_df["CHURNED"].notna()
].copy()

print("Dataset shape:", modeling_df.shape)
print("Unique customers:", modeling_df["CUSTOMER_ID"].nunique())
print("\nChurn counts:")
print(modeling_df["CHURNED"].value_counts())
print("\nChurn percentages:")
print((modeling_df["CHURNED"].value_counts(normalize=True) * 100).round(2))
print("\nTop missing-value counts:")
print(modeling_df.isna().sum().sort_values(ascending=False).head(10))

modeling_df.to_csv(
    "/content/customer_level_modeling_dataset.csv",
    index=False
)

print("\nSaved customer_level_modeling_dataset.csv")

**Prepare Features for Model Training**

In [ ]:

data = modeling_df[
    modeling_df["INSUFFICIENT_HISTORY_FLAG"] == 0
].copy()

excluded_cols = [
    "CUSTOMER_ID",
    "CHURNED",
    "INSUFFICIENT_HISTORY_FLAG",
    "OBSERVATION_END_DATE",
    "LAST_PURCHASE_DATE",
    "LAST_ACTIVE_MONTH",
    "MONTHS_OBSERVED"
]

X = data.drop(
    columns=[col for col in excluded_cols if col in data.columns]
).copy()

y = data["CHURNED"].astype(int)

date_cols = X.select_dtypes(include=["datetime", "datetimetz"]).columns
X = X.drop(columns=date_cols)

numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in categorical_cols:
    X[col] = X[col].fillna("Unknown").astype(str)


print("Feature matrix:", X.shape)
print("Target size:", y.shape)
print("Numeric features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))
print("\nTarget distribution:")
print(y.value_counts())

In [ ]:
import pandas as pd
import numpy as np

X_temp = X.copy()
X_temp['CUSTOMER_ID'] = data['CUSTOMER_ID']
X_temp = X_temp.set_index('CUSTOMER_ID')

y_temp = pd.Series(y.values, index=data['CUSTOMER_ID'])

cohort_mapping = data[['CUSTOMER_ID']].merge(
    profile_clean[["CUSTOMER_ID", "SIGNUP_DATE"]],
    on="CUSTOMER_ID",
    how="left"
)

split_date = pd.Timestamp("2024-01-01")

train_ids = cohort_mapping.loc[cohort_mapping["SIGNUP_DATE"] < split_date, "CUSTOMER_ID"]
test_ids = cohort_mapping.loc[cohort_mapping["SIGNUP_DATE"] >= split_date, "CUSTOMER_ID"]

X_train = X_temp.loc[train_ids]
y_train = y_temp.loc[train_ids]

X_test = X_temp.loc[test_ids]
y_test = y_temp.loc[test_ids]

print(f"Cohort Split Date: {split_date.date()}")
print(f"Training set (Signed up before split): {X_train.shape[0]} customers")
print(f"Testing set (Signed up after split): {X_test.shape[0]} customers")
print(f"\nTraining Churn Rate: {y_train.mean():.2%}")
print(f"Testing Churn Rate: {y_test.mean():.2%}")


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


**Build Data Preprocessing Pipeline**

In [ ]:

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Preprocessing pipeline created successfully.")

**Define Machine Learning Models**

In [ ]:

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}
print("Models ready:", ", ".join(models.keys()))

Train and Evaluate Churn Prediction Models

In [ ]:

trained_models = {}
results = {}
test_predictions = {}

from sklearn.metrics import classification_report

for name, classifier in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", classifier)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1_Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) > 1 else np.nan,
        "PR_AUC": average_precision_score(y_test, y_prob) if len(np.unique(y_test)) > 1 else np.nan
    }

    trained_models[name] = pipeline
    test_predictions[name] = {
        "labels": y_pred,
        "probabilities": y_prob
    }

    print(f"\n{name}")
    print(pd.Series(results[name]).round(4))
    print("\nClassification report:")
    print(classification_report(
        y_test, y_pred,
        target_names=["Not Churned", "Churned"] if len(np.unique(y_test)) > 1 else ["Not Churned"],
        zero_division=0
    ))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Not Churned", "Churned"] if len(np.unique(y_test)) > 1 else ["Not Churned"]
    ).plot(values_format="d")

    plt.title(f"Confusion Matrix - {name} (Cohort Split)")
    plt.tight_layout()

    safe_name = name.lower().replace(" ", "_")
    plt.savefig(
        f"/content/confusion_matrix_{safe_name}.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

print("\nModel training and evaluation on cohort split completed.")

Model Performance Generator


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

smote_rf_pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

smote_rf_pipeline.fit(X_train, y_train)

y_pred_opt = smote_rf_pipeline.predict(X_test)
y_prob_opt = smote_rf_pipeline.predict_proba(X_test)[:, 1]

results["SMOTE + Random Forest"] = {
    "Accuracy": accuracy_score(y_test, y_pred_opt),
    "Precision": precision_score(y_test, y_pred_opt, zero_division=0),
    "Recall": recall_score(y_test, y_pred_opt, zero_division=0),
    "F1_Score": f1_score(y_test, y_pred_opt, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_prob_opt) if len(np.unique(y_test)) > 1 else np.nan,
    "PR_AUC": average_precision_score(y_test, y_prob_opt) if len(np.unique(y_test)) > 1 else np.nan
}

print("SMOTE + Random Forest Performance on Cohort Split:")
for metric, val in results["SMOTE + Random Forest"].items():
    print(f"{metric}: {val:.4%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_opt, target_names=["Not Churned", "Churned"], zero_division=0))

**Compare Model Performance**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

metrics = ["Accuracy", "Precision", "Recall", "F1_Score"]
comparison = pd.DataFrame(results).T[metrics] * 100

ax = comparison.plot(kind="bar", figsize=(12, 7), width=0.8)

plt.title("Model Performance Comparison (All Models)", fontsize=14, fontweight="bold")
plt.xlabel("Model", fontsize=12)
plt.ylabel("Performance (%)", fontsize=12)
plt.ylim(0, 110)
plt.yticks(np.arange(0, 101, 10))
plt.xticks(rotation=0, fontsize=11)
plt.legend(["Accuracy", "Precision", "Recall", "F1-score"], loc="lower left")
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bars in ax.containers:
    ax.bar_label(
        bars,
        labels=[f"{bar.get_height():.1f}%" for bar in bars],
        padding=3,
        fontsize=8,
        fontweight="bold"
    )

plt.tight_layout()
plt.savefig("/content/model_performance_comparison.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

auc_metrics = ["ROC_AUC", "PR_AUC"]
auc_comparison = pd.DataFrame(results).T[auc_metrics] * 100

ax = auc_comparison.plot(kind="bar", figsize=(10, 6), width=0.6, color=["#1f77b4", "#ff7f0e"])

plt.title("AUC Performance Comparison (ROC-AUC vs. PR-AUC)", fontsize=14, fontweight="bold")
plt.xlabel("Model", fontsize=12)
plt.ylabel("Performance Score (%)", fontsize=12)
plt.ylim(80, 105)
plt.yticks(np.arange(80, 101, 5))
plt.xticks(rotation=0, fontsize=11)
plt.legend(["ROC-AUC", "PR-AUC"], loc="lower left", fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bars in ax.containers:
    ax.bar_label(
        bars,
        labels=[f"{bar.get_height():.2f}%" for bar in bars],
        padding=3,
        fontsize=9,
        fontweight="bold"
    )

plt.tight_layout()
plt.savefig("/content/auc_performance_comparison.png", dpi=300)
plt.show()

Generate Test Customer Churn Predictions

In [ ]:

results_df = pd.DataFrame(results).T
best_model_name = results_df["PR_AUC"].idxmax()
best_model = trained_models[best_model_name]

print("Selected model:", best_model_name)

best_prob = best_model.predict_proba(X_test)[:, 1]
best_pred = best_model.predict(X_test)

predictions_df = pd.DataFrame({
    "CUSTOMER_ID": data.loc[X_test.index, "CUSTOMER_ID"].values,
    "ACTUAL_CHURN": y_test.values,
    "CHURN_PROBABILITY": best_prob,
    "PREDICTED_CHURN": best_pred
})


predictions_df["RISK_LEVEL"] = pd.cut(
    predictions_df["CHURN_PROBABILITY"],
    bins=[-0.001, 0.30, 0.60, 1.0],
    labels=["Low", "Medium", "High"]
)

predictions_df = predictions_df.sort_values(
    "CHURN_PROBABILITY", ascending=False
)

display(predictions_df.head(10))

predictions_df.to_csv(
    "/content/test_customer_churn_predictions.csv",
    index=False
)

print("Test predictions saved.")

Save the Best Churn Prediction Model

In [ ]:

joblib.dump(best_model, "/content/best_churn_model.joblib")

print("Best model saved successfully.")

Package and Download Model Outputs

In [ ]:

results_df = pd.DataFrame(results).T
results_df.index.name = "Model"
results_df = results_df.reset_index()

results_df.to_csv(
    "/content/model_evaluation_results.csv",
    index=False
)

files_to_package = [
    "/content/model_evaluation_results.csv",
    "/content/test_customer_churn_predictions.csv",
    "/content/best_churn_model.joblib",
    "/content/confusion_matrix_logistic_regression.png",
    "/content/confusion_matrix_random_forest.png"
]

zip_path = "/content/customer_churn_model_outputs.zip"

with zipfile.ZipFile(zip_path, "w") as zipf:
    for path in files_to_package:
        if os.path.exists(path):
            zipf.write(path, arcname=os.path.basename(path))

print("ZIP file created:", zip_path)
files.download(zip_path)

In [ ]:

display(results_df.round(4))

print("Best model:", best_model_name)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Identify Important Churn Predictors

In [ ]:

importance_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": importance_result.importances_mean,
    "Std": importance_result.importances_std
}).sort_values("Importance", ascending=False)

print("Top 15 churn predictors:")
display(importance_df.head(15).round(4))

top_features = importance_df.head(15).sort_values("Importance")

plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Decrease in PR-AUC when feature is shuffled")
plt.title("Top 15 Churn Predictors")
plt.tight_layout()
plt.savefig("/content/feature_importance.png", dpi=300)
plt.show()

importance_df.to_csv("/content/feature_importance.csv", index=False)

Churn Analysis by Customer Segment

In [ ]:

segment_data = modeling_df.merge(
    profile_clean[["CUSTOMER_ID", "SIGNUP_DATE"]],
    on="CUSTOMER_ID",
    how="left",
    validate="one_to_one"
)

segment_data["SIGNUP_DATE"] = pd.to_datetime(
    segment_data["SIGNUP_DATE"], errors="coerce"
)

def churn_summary(df, column):
    result = df.groupby(column, dropna=False).agg(
        Customers=("CHURNED", "size"),
        Churn_Rate=("CHURNED", "mean")
    ).reset_index()

    result["Churn_Rate"] *= 100
    return result

tier_analysis = churn_summary(segment_data, "MEMBERSHIP_TIER")
city_analysis = churn_summary(segment_data, "CITY").sort_values(
    "Churn_Rate", ascending=False
)

segment_data["SIGNUP_COHORT"] = (
    segment_data["SIGNUP_DATE"].dt.to_period("Q").astype("string")
)

cohort_analysis = churn_summary(
    segment_data, "SIGNUP_COHORT"
).sort_values("SIGNUP_COHORT")

print("Churn by membership tier:")
display(tier_analysis.round(2))

print("Churn by city:")
display(city_analysis.round(2))

print("Churn by signup cohort:")
display(cohort_analysis.round(2))

tier_analysis.to_csv("/content/churn_by_tier.csv", index=False)
city_analysis.to_csv("/content/churn_by_city.csv", index=False)
cohort_analysis.to_csv("/content/churn_by_signup_cohort.csv", index=False)

Ablation Study


In [ ]:

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score
)

demographic_cols = [
    "AGE", "GENDER", "CITY", "MEMBERSHIP_TIER",
    "PREFERRED_CATEGORY", "HOME_STORE_PRICE_TIER",
    "TENURE_DAYS"
]

behaviour_cols = [
    "HIST_TRANSACTIONS", "HIST_TOTAL_SPEND",
    "HIST_ACTIVE_MONTHS", "RECENCY_DAYS",
    "RECENT3_TRANSACTIONS", "RECENT3_SPEND",
    "PREV3_TRANSACTIONS", "PREV3_SPEND"
]

engagement_cols = [
    "HIST_APP_SESSIONS", "HIST_EMAILS_OPENED",
    "HIST_COUPONS_REDEEMED", "HIST_PROMO_TXN_PCT_MEAN",
    "RECENT3_APP_SESSIONS", "RECENT3_EMAILS_OPENED",
    "PREV3_APP_SESSIONS", "PREV3_EMAILS_OPENED"
]

support_cols = [
    "HIST_SUPPORT_TICKETS", "HIST_COMPLAINT_MONTHS",
    "RECENT3_SUPPORT_TICKETS", "PREV3_SUPPORT_TICKETS"
]

groups = {
    "Demographics only": demographic_cols,
    "Behaviour only": behaviour_cols,
    "Engagement + Support": engagement_cols + support_cols,
    "All features": demographic_cols + behaviour_cols +
                    engagement_cols + support_cols
}

X_ab = X.copy()
y_ab = y.copy()

X_train_ab, X_test_ab, y_train_ab, y_test_ab = train_test_split(
    X_ab, y_ab,
    test_size=0.20,
    random_state=42,
    stratify=y_ab
)

ablation_results = []

for group_name, requested_cols in groups.items():
    cols = [c for c in requested_cols if c in X_ab.columns]

    if not cols:
        print(f"Skipping {group_name}: no matching columns.")
        continue

    num_cols = X_train_ab[cols].select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()
    cat_cols = [c for c in cols if c not in num_cols]

    prep = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

    model = Pipeline([
        ("preprocessor", prep),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(X_train_ab[cols], y_train_ab)
    pred = model.predict(X_test_ab[cols])
    prob = model.predict_proba(X_test_ab[cols])[:, 1]

    ablation_results.append({
        "Feature_Group": group_name,
        "Features_Used": len(cols),
        "Accuracy": accuracy_score(y_test_ab, pred),
        "Precision": precision_score(y_test_ab, pred, zero_division=0),
        "Recall": recall_score(y_test_ab, pred, zero_division=0),
        "F1": f1_score(y_test_ab, pred, zero_division=0),
        "PR_AUC": average_precision_score(y_test_ab, prob)
    })

ablation_df = pd.DataFrame(ablation_results).sort_values(
    "PR_AUC", ascending=False
)

display(ablation_df.round(4))
ablation_df.to_csv("/content/ablation_study_results.csv", index=False)

print("Saved: /content/ablation_study_results.csv")


Retention Scenario Analysis

In [ ]:

scenario_df = pd.DataFrame({
    "CUSTOMER_ID": data.loc[X_test.index, "CUSTOMER_ID"].values,
    "ACTUAL_CHURN": y_test.values,
    "BASELINE_RISK": best_model.predict_proba(X_test)[:, 1]
})

risk_cutoff = scenario_df["BASELINE_RISK"].quantile(0.80)
scenario_df["TARGETED"] = scenario_df["BASELINE_RISK"] >= risk_cutoff

assumed_risk_reduction = 0.20

scenario_df["SCENARIO_RISK"] = scenario_df["BASELINE_RISK"]
scenario_df.loc[scenario_df["TARGETED"], "SCENARIO_RISK"] = (
    scenario_df.loc[scenario_df["TARGETED"], "BASELINE_RISK"]
    * (1 - assumed_risk_reduction)
)

baseline_expected = scenario_df["BASELINE_RISK"].sum()
scenario_expected = scenario_df["SCENARIO_RISK"].sum()

targeted_count = int(scenario_df["TARGETED"].sum())
risk_difference = baseline_expected - scenario_expected

print("===== RETENTION SCENARIO =====")
print("Customers evaluated:", len(scenario_df))
print("Customers targeted:", targeted_count)
print("Targeting percentage:",
      round(100 * scenario_df["TARGETED"].mean(), 2), "%")
print("Assumed risk reduction for targeted customers:",
      assumed_risk_reduction * 100, "%")

print("\nSum of predicted churn probabilities (baseline):",
      round(baseline_expected, 2))
print("Sum after simulated intervention:",
      round(scenario_expected, 2))
print("Estimated difference in expected churn cases:",
      round(risk_difference, 2))

print("\nMean baseline predicted risk:",
      round(scenario_df["BASELINE_RISK"].mean() * 100, 2), "%")
print("Mean scenario predicted risk:",
      round(scenario_df["SCENARIO_RISK"].mean() * 100, 2), "%")

scenario_df.to_csv(
    "/content/retention_scenario_predictions.csv",
    index=False
)

print("\nSaved: /content/retention_scenario_predictions.csv")
